<a href="https://colab.research.google.com/github/2k25cse2512541-wq/assignment1-flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2k25cse2512541-wq/assignment1-flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit of Analysis (Grain):** **1 row = 1 unique content item (`content_hash_id`) per client (`client_hash_id`) on a specific daily report date (`report_date`)**.
* **Iteration Time Window:** **March 1, 2026 – March 31, 2026 (`month=2026-03`)**. We use this mid-panel month for feature engineering and validation.
* **Sealed Test Window:** **June 2026 (`_sample` table)** is strictly reserved as a sealed test set and is excluded from feature development to avoid temporal leakage.

In [8]:
import duckdb
from google.colab import userdata

# 1. Fetch HF READ token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Connect DuckDB and register Hugging Face credentials
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 3. Define the mid-panel iteration month path using the hf:// protocol
mid_panel_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# 4. Query 1: Verify Grain (Should return 0 duplicate rows)
grain_check_query = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
FROM read_parquet('{mid_panel_path}')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
"""
duplicates = con.sql(grain_check_query).df()
print(f"Duplicate rows violating grain: {len(duplicates)}")

# 5. Query 2: Verify Time Window (Date Bounds)
time_window_query = f"""
SELECT
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(*) AS total_rows
FROM read_parquet('{mid_panel_path}')
"""
time_window_df = con.sql(time_window_query).df()
print("\n--- Time Window Verification ---")
print(time_window_df.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows violating grain: 0

--- Time Window Verification ---
min_report_date max_report_date  total_rows
     2026-03-01      2026-03-31     9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
### Field Categorization

* **Context (Identifiers & Metadata):**
  * `report_date`: Daily timestamp.
  * `client_hash_id`: Unique client identifier.
  * `content_hash_id`: Unique page/URL identifier.

* **Features (Knowable at Decision Time):**
  * `gsc_impressions`: Daily search impressions.
  * `gsc_clicks`: Daily user clicks.
  * `gsc_avg_position`: Daily average rank position.
  * `gsc_data_available`: Flag indicating if GSC data was fetched successfully.

* **Label (Target Variable):**
  * `is_declining_label`: Binary target (1 if traffic drops significantly, 0 otherwise).

* **Excluded:**
  * **Future performance metrics** (e.g., clicks from a future window). Excluded to prevent temporal label leakage, as future data is impossible to know at prediction time.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify our claimed fields exist and check for missing/null values
missing_values_query = f"""
SELECT
    -- Check total row counts vs non-null counts to find missing data
    COUNT(*) - COUNT(report_date) AS missing_report_date,
    COUNT(*) - COUNT(client_hash_id) AS missing_client_id,
    COUNT(*) - COUNT(content_hash_id) AS missing_content_id,
    COUNT(*) - COUNT(gsc_impressions) AS missing_gsc_impressions,
    COUNT(*) - COUNT(gsc_clicks) AS missing_gsc_clicks,
    COUNT(*) - COUNT(gsc_avg_position) AS missing_gsc_avg_position
FROM read_parquet('{mid_panel_path}')
"""

print("--- Missing Values Check (Should be 0 if data is clean) ---")
missing_df = con.sql(missing_values_query).df()
print(missing_df.T.rename(columns={0: 'null_count'}))

# Verify the numeric ranges (sanity check for our features)
summary_query = f"""
SELECT
    MIN(gsc_impressions) as min_impressions,
    MAX(gsc_impressions) as max_impressions,
    MIN(gsc_avg_position) as min_position,
    MAX(gsc_avg_position) as max_position
FROM read_parquet('{mid_panel_path}')
"""

print("\n--- Feature Bounds Verification ---")
print(con.sql(summary_query).df().to_string(index=False))

--- Missing Values Check (Should be 0 if data is clean) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                          null_count
missing_report_date                0
missing_client_id                  0
missing_content_id                 0
missing_gsc_impressions            0
missing_gsc_clicks                 0
missing_gsc_avg_position     6230317

--- Feature Bounds Verification ---
 min_impressions  max_impressions  min_position  max_position
               0            40084           0.0         498.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Key Data Limitations

1. **Undefined Position on Zero Impressions:**
   * `gsc_avg_position` is `NULL` for ~63% of daily rows because Google Search Console cannot record a rank position when `gsc_impressions = 0`. Imputing these with `0` would falsely imply top search ranking.

2. **Integration Gaps (`gsc_data_available` / `ga4_data_available`):**
   * Days where a client's GSC or GA4 integration was inactive reflect missing API data rather than true zero traffic. Models must distinguish between "zero clicks" and "uncollected data."

3. **Survivorship Bias:**
   * The dataset only tracks pages that were crawled and logged by search/analytics platforms. Pages that were published but never indexed by search engines are omitted entirely.

In [10]:
# 1. Prove why gsc_avg_position is NULL (tied to zero impressions)
null_pos_query = f"""
SELECT
    COUNT(*) AS total_null_position_rows,
    COUNT(CASE WHEN gsc_impressions = 0 THEN 1 END) AS null_pos_where_impressions_zero,
    ROUND(COUNT(CASE WHEN gsc_impressions = 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_zero_impressions
FROM read_parquet('{mid_panel_path}')
WHERE gsc_avg_position IS NULL
"""

print("--- Limitation 1: NULL Position Cause ---")
print(con.sql(null_pos_query).df().to_string(index=False))

# 2. Check integration availability flags across the month
integration_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN gsc_data_available IS FALSE THEN 1 END) AS gsc_unavailable_rows,
    COUNT(CASE WHEN ga4_data_available IS FALSE THEN 1 END) AS ga4_unavailable_rows
FROM read_parquet('{mid_panel_path}')
"""

print("\n--- Limitation 2: Integration Data Gaps ---")
print(con.sql(integration_query).df().to_string(index=False))

--- Limitation 1: NULL Position Cause ---
 total_null_position_rows  null_pos_where_impressions_zero  pct_zero_impressions
                  6230317                          6230317                 100.0

--- Limitation 2: Integration Data Gaps ---
 total_rows  gsc_unavailable_rows  ga4_unavailable_rows
    9841378               6230317               6408671


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.